# Orders Silver

### Imorting Library

In [0]:
from pyspark.sql.functions import *

### Assiging static data 

In [0]:
bronze_orders_df = spark.table("ecommerce_lakehouse.bronze.orders_raw")
display(bronze_orders_df)

### Printing schema 

In [0]:
bronze_orders_df.printSchema()

### Changing dateTime columns

In [0]:
silver_orders_df = bronze_orders_df \
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")
    ) \
    .withColumn(
        "order_approved_at",
        to_timestamp("order_approved_at")
    ) \
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date")
    ) \
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")
    ) \
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")
    )
    

In [0]:
display(silver_orders_df)

### Removing Null values from critical columns

In [0]:
silver_orders_df = silver_orders_df.dropna(
    subset=[
        "order_id",
        "customer_id",
        "order_purchase_timestamp"
    ]
)


### Removing duplicates from our primary key column

In [0]:
silver_orders_df = silver_orders_df.dropDuplicates(
    ["order_id"]
)

### Derived columns

In [0]:
silver_orders_df = silver_orders_df.withColumn(
    "purchase_date",
    to_date("order_purchase_timestamp")
)

In [0]:
silver_orders_df = silver_orders_df.withColumn(
    "purchase_year",
    year("order_purchase_timestamp")
)

In [0]:
silver_orders_df = silver_orders_df.withColumn(
    "purchase_month",
    month("order_purchase_timestamp")
)

In [0]:
silver_orders_df = silver_orders_df.withColumn(
    "delivery_days",
    datediff(
        "order_delivered_customer_date",
        "order_purchase_timestamp")
)

In [0]:
silver_orders_df = silver_orders_df.withColumn(
    "is_delivered",
    when(
        col("order_status") == "delivered",
        True
    ).otherwise(False)
)

### Reorder columns

In [0]:
silver_orders_df = silver_orders_df.select(
   "order_id",
   "customer_id",
   "order_status",
   "order_purchase_timestamp",
   "order_approved_at",
   "order_delivered_carrier_date",
   "order_delivered_customer_date",
   "order_estimated_delivery_date",
   "purchase_date",
   "purchase_year",
   "purchase_month",
   "delivery_days",
   "is_delivered",
   "ingestion_timestamp",
   "load_date",
   "source_file"
)

In [0]:
(
    silver_orders_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            "ecommerce_lakehouse.silver.orders_clean"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.silver.orders_clean
LIMIT 10;